In [ ]:
import pandas as pd
import zipfile
import os

# Upload the exported ZIP file
from google.colab import files
uploaded = files.upload()

zip_path = next(iter(uploaded))

# Extract the datasets
extract_dir = "/content/wasteloop_audit_exports"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

# Load both datasets
train_df = pd.read_csv(
    f"{extract_dir}/train_audited.csv"
)

test_df = pd.read_csv(
    f"{extract_dir}/test_audited.csv"
)

print("Training dataset:", train_df.shape)
print("Testing dataset:", test_df.shape)

print("\nTraining columns:")
print(train_df.columns.tolist())

print("\nTesting columns:")
print(test_df.columns.tolist())

Saving WasteLoop_Audited_Source.zip to WasteLoop_Audited_Source (1).zip
Training dataset: (21528, 8)
Testing dataset: (5382, 7)

Training columns:
['Unnamed: 0', 'Phrase', 'Class', 'Class_index', 'normalized_phrase', 'word_count', 'opening_pattern', 'Phrase_norm']

Testing columns:
['Unnamed: 0', 'Phrase', 'Class', 'Class_index', 'normalized_phrase', 'opening_pattern', 'opening_seen_in_train']


In [ ]:
# Work on a copy; preserve the original audited dataset
working_df = train_df.copy()

# Use the normalized phrase key from our earlier audit
working_df["phrase_key"] = (
    working_df["normalized_phrase"].astype(str).str.strip()
)

# Identify phrases assigned to more than one class
label_counts = working_df.groupby("phrase_key")["Class"].nunique()

conflict_keys = label_counts[label_counts > 1].index

# Quarantine contradictory examples
conflict_quarantine_df = working_df[
    working_df["phrase_key"].isin(conflict_keys)
].copy()

# Retain only phrases with consistent labels
consistent_df = working_df[
    ~working_df["phrase_key"].isin(conflict_keys)
].copy()

# Remove repeated phrases with the same label
train_clean = consistent_df.drop_duplicates(
    subset="phrase_key",
    keep="first"
).copy()

# Remove our temporary helper column
train_clean.drop(columns=["phrase_key"], inplace=True)
conflict_quarantine_df.drop(columns=["phrase_key"], inplace=True)

print("Original training records:", len(train_df))
print("Conflicting records quarantined:", len(conflict_quarantine_df))
print("Clean training records:", len(train_clean))
print("Unique classes retained:", train_clean["Class"].nunique())
print("Remaining duplicate normalized phrases:",
      train_clean["normalized_phrase"].duplicated().sum())

Original training records: 21528
Conflicting records quarantined: 821
Clean training records: 20002
Unique classes retained: 50
Remaining duplicate normalized phrases: 0


In [ ]:
from sklearn.model_selection import train_test_split

X = train_clean["Phrase"]
y = train_clean["Class"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

print("\nClasses in training:", y_train.nunique())
print("Classes in validation:", y_val.nunique())

print("\nTraining distribution:")
print(y_train.value_counts().describe())

print("\nValidation distribution:")
print(y_val.value_counts().describe())

Training samples: 16001
Validation samples: 4001

Classes in training: 50
Classes in validation: 50

Training distribution:
count     50.00000
mean     320.02000
std      174.04169
min      111.00000
25%      194.00000
50%      248.00000
75%      390.75000
max      906.00000
Name: count, dtype: float64

Validation distribution:
count     50.000000
mean      80.020000
std       43.493133
min       27.000000
25%       48.250000
50%       62.000000
75%       97.750000
max      226.000000
Name: count, dtype: float64


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Build the baseline NLP pipeline
tfidf_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train the model
tfidf_model.fit(X_train, y_train)

# Inspect the learned feature space
vectorizer = tfidf_model.named_steps["tfidf"]

print("Model training completed.")
print("TF-IDF vocabulary size:", len(vectorizer.vocabulary_))
print("Number of training samples:", len(X_train))
print("Number of target classes:", len(tfidf_model.classes_))

Model training completed.
TF-IDF vocabulary size: 3178
Number of training samples: 16001
Number of target classes: 50


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Predict on unseen validation samples
y_pred = tfidf_model.predict(X_val)

# Calculate evaluation metrics
accuracy = accuracy_score(y_val, y_pred)

precision_macro = precision_score(
    y_val, y_pred, average="macro", zero_division=0
)

recall_macro = recall_score(
    y_val, y_pred, average="macro", zero_division=0
)

f1_macro = f1_score(
    y_val, y_pred, average="macro", zero_division=0
)

f1_weighted = f1_score(
    y_val, y_pred, average="weighted", zero_division=0
)

print("TF-IDF + Logistic Regression — Validation Results")
print("-" * 50)
print(f"Accuracy:          {accuracy:.4f}")
print(f"Macro Precision:   {precision_macro:.4f}")
print(f"Macro Recall:      {recall_macro:.4f}")
print(f"Macro F1:          {f1_macro:.4f}")
print(f"Weighted F1:       {f1_weighted:.4f}")

print("\nDetailed classification report:")
print(classification_report(
    y_val, y_pred, zero_division=0
))

TF-IDF + Logistic Regression — Validation Results
--------------------------------------------------
Accuracy:          0.9878
Macro Precision:   0.9880
Macro Recall:      0.9821
Macro F1:          0.9844
Weighted F1:       0.9876

Detailed classification report:
                               precision    recall  f1-score   support

                ALUMINIUM CAN       1.00      1.00      1.00       193
              ALUMINIUM SHEET       1.00      1.00      1.00        37
               ALUMINIUM TRAY       1.00      1.00      1.00        37
               CIGARETTE BUTT       1.00      1.00      1.00        57
               CIGARETTE PACK       1.00      1.00      1.00       109
        COMPOSTABLE PACKAGING       0.76      0.76      0.76        59
            CONDIMENT PACKETS       1.00      1.00      1.00       134
                   COVID TEST       1.00      1.00      1.00        36
              CRUMBLED TISSUE       1.00      1.00      1.00        61
          CYLINDRICAL BAT

In [ ]:
# Build a table of incorrect predictions

error_df = pd.DataFrame({
    "Phrase": X_val,
    "Actual": y_val,
    "Predicted": y_pred
})

error_df = error_df[
    error_df["Actual"] != error_df["Predicted"]
].copy()

print("Total validation samples:", len(y_val))
print("Incorrect predictions:", len(error_df))
print("Correct predictions:", len(y_val) - len(error_df))

print("\nMost frequent confusion pairs:")
confusion_pairs = (
    error_df.groupby(["Actual", "Predicted"])
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)

display(confusion_pairs.head(15))

print("\nExamples of incorrect predictions:")
display(error_df.head(15))

Total validation samples: 4001
Incorrect predictions: 49
Correct predictions: 3952

Most frequent confusion pairs:


,Actual,Predicted,Count
0,COMPOSTABLE PACKAGING,PAPER PACKAGING,14
1,PAPER PACKAGING,COMPOSTABLE PACKAGING,14
5,PLASTIC PACKAGING,PLASTIC DISH,8
4,PLASTIC CUP,PAPER CUP,8
2,PLASTIC BOWL,PAPER BOWL,4
3,PLASTIC BOWL,PLASTIC DISH,1



Examples of incorrect predictions:


,Phrase,Actual,Predicted
13283,In which bin should I throw my sustainable pac...,PAPER PACKAGING,COMPOSTABLE PACKAGING
4972,Where do I have to throw a compostable packagi...,COMPOSTABLE PACKAGING,PAPER PACKAGING
3493,Where do I throw my compostable packaging,COMPOSTABLE PACKAGING,PAPER PACKAGING
7571,Can I recycle my disposable bowl ?,PLASTIC BOWL,PAPER BOWL
4524,Where can I put a compostable packaging ?,COMPOSTABLE PACKAGING,PAPER PACKAGING
9442,"I have a disposable cup to throw, where should...",PLASTIC CUP,PAPER CUP
12878,what bin is best suited for my biodegradable p...,PAPER PACKAGING,COMPOSTABLE PACKAGING
10517,I would like to throw a eco-friendly packaging,COMPOSTABLE PACKAGING,PAPER PACKAGING
5464,I have my compostable packaging,COMPOSTABLE PACKAGING,PAPER PACKAGING
2726,"Excuse me, can you help me ? I have to throw m...",PLASTIC PACKAGING,PLASTIC DISH


In [ ]:
import numpy as np

# Get class probabilities for validation examples
val_probabilities = tfidf_model.predict_proba(X_val)

prob_df = pd.DataFrame(
    val_probabilities,
    columns=tfidf_model.classes_,
    index=X_val.index
)

# Calculate predicted and actual class probabilities for errors
error_df["Predicted_confidence"] = [
    prob_df.loc[idx, row["Predicted"]]
    for idx, row in error_df.iterrows()
]

error_df["Actual_class_probability"] = [
    prob_df.loc[idx, row["Actual"]]
    for idx, row in error_df.iterrows()
]

error_df = error_df.sort_values(
    "Predicted_confidence", ascending=False
)

print("Average confidence on incorrect predictions:",
      round(error_df["Predicted_confidence"].mean(), 4))

print("\nIncorrect predictions with highest confidence:")
display(error_df[
    ["Phrase", "Actual", "Predicted",
     "Predicted_confidence", "Actual_class_probability"]
].head(15))

Average confidence on incorrect predictions: 0.4418

Incorrect predictions with highest confidence:


,Phrase,Actual,Predicted,Predicted_confidence,Actual_class_probability
3686,Where to throw my biodegradable packaging ?,PAPER PACKAGING,COMPOSTABLE PACKAGING,0.690710,0.184121
8778,Where do I put my biodegradable packaging ?,PAPER PACKAGING,COMPOSTABLE PACKAGING,0.656864,0.158204
17921,Is this disposable cup recyclable ?,PLASTIC CUP,PAPER CUP,0.620248,0.194173
6773,Where does my disposable cup belong?,PLASTIC CUP,PAPER CUP,0.592448,0.206203
20217,Could I throw a disposable cup in this trash ?,PLASTIC CUP,PAPER CUP,0.588247,0.176923
9442,"I have a disposable cup to throw, where should...",PLASTIC CUP,PAPER CUP,0.577314,0.172002
7863,I have my biodegradable packaging to throw,PAPER PACKAGING,COMPOSTABLE PACKAGING,0.545103,0.305034
94,"I have my disposable cup to throw, where shoul...",PLASTIC CUP,PAPER CUP,0.541788,0.209051
2576,Where does my rest of disposable cup go ?,PLASTIC CUP,PAPER CUP,0.536064,0.141861
4980,Where do I have to throw a recyclable packaging ?,PAPER PACKAGING,COMPOSTABLE PACKAGING,0.530100,0.252064


In [ ]:
import joblib
import json
import os
import zipfile
from sklearn.metrics import classification_report

# Create an output directory
output_dir = "/content/WasteLoop_Baseline"
os.makedirs(output_dir, exist_ok=True)

# 1. Save the complete TF-IDF + Logistic Regression pipeline
joblib.dump(
    tfidf_model,
    f"{output_dir}/tfidf_logistic_regression.joblib"
)

# 2. Save evaluation metrics and experiment details
results = {
    "model": "TF-IDF + Logistic Regression",
    "accuracy": float(accuracy),
    "macro_precision": float(precision_macro),
    "macro_recall": float(recall_macro),
    "macro_f1": float(f1_macro),
    "weighted_f1": float(f1_weighted),
    "training_samples": len(X_train),
    "validation_samples": len(X_val),
    "classes": int(y.nunique()),
    "tfidf_vocabulary_size": len(vectorizer.vocabulary_),
    "tfidf_ngram_range": [1, 2],
    "random_state": 42
}

with open(f"{output_dir}/validation_metrics.json", "w") as f:
    json.dump(results, f, indent=4)

# Save detailed per-class evaluation
report = classification_report(
    y_val, y_pred, output_dict=True, zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    f"{output_dir}/classification_report.csv"
)

# Save incorrect predictions for future analysis
error_df.to_csv(
    f"{output_dir}/validation_errors.csv",
    index=False
)

# Package all artifacts into one ZIP
zip_path = "/content/WasteLoop_TFIDF_Baseline_Results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in os.listdir(output_dir):
        z.write(
            os.path.join(output_dir, filename),
            arcname=filename
        )

print("Baseline artifacts saved successfully.")
print("\nFiles:")
for filename in os.listdir(output_dir):
    print("-", filename)

print("\nValidation accuracy:", round(accuracy * 100, 2), "%")

from google.colab import files
files.download(zip_path)

Baseline artifacts saved successfully.

Files:
- classification_report.csv
- tfidf_logistic_regression.joblib
- validation_errors.csv
- validation_metrics.json

Validation accuracy: 98.78 %


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# I am preparing a template for my independent test dataset.

classes = sorted(train_clean["Class"].unique())

test_rows = []

for class_name in classes:
    for example_number in range(1, 6):
        test_rows.append({
            "test_id": f"WW{len(test_rows) + 1:04d}",
            "phrase": "",
            "expected_class": class_name,
            "case_type": "clear",
            "expected_action": "classify",
            "source_description": "",
            "annotation_notes": ""
        })

independent_test_template = pd.DataFrame(test_rows)

template_path = "/content/WasteLoop_Independent_Test_Template.csv"

independent_test_template.to_csv(
    template_path,
    index=False
)

print("Independent test template created.")
print("Number of classes:", len(classes))
print("Examples allocated per class:", 5)
print("Total empty test records:", len(independent_test_template))

from google.colab import files
files.download(template_path)

Independent test template created.
Number of classes: 50
Examples allocated per class: 5
Total empty test records: 250


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import re
from google.colab import files

# I am loading my independently prepared test dataset.
uploaded = files.upload()
file_name = next(iter(uploaded))

independent_test_df = pd.read_csv(file_name)

# I am checking that all required columns are present.
required_columns = [
    "test_id", "phrase", "expected_class",
    "case_type", "expected_action"
]

assert all(
    col in independent_test_df.columns
    for col in required_columns
), "Some required columns are missing."

# I am normalizing the text consistently before comparing phrases.
def normalize_for_overlap(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

independent_test_df["phrase_key"] = (
    independent_test_df["phrase"].apply(normalize_for_overlap)
)

train_keys = set(
    train_df["Phrase"].apply(normalize_for_overlap)
)

test_keys = set(
    test_df["Phrase"].apply(normalize_for_overlap)
)

# I am checking for duplicate phrases and source-data overlap.
duplicate_count = independent_test_df["phrase_key"].duplicated().sum()

train_overlap = independent_test_df[
    independent_test_df["phrase_key"].isin(train_keys)
]

test_overlap = independent_test_df[
    independent_test_df["phrase_key"].isin(test_keys)
]

# I am checking whether the expected labels match my source classes.
source_classes = set(train_df["Class"].unique())
new_classes = set(independent_test_df["expected_class"].unique())

print("INDEPENDENT TEST DATA AUDIT")
print("-" * 45)

print("Total examples:", len(independent_test_df))
print("Unique classes:", independent_test_df["expected_class"].nunique())
print("Duplicate phrases within new dataset:", duplicate_count)

print("\nClass count distribution:")
display(independent_test_df["expected_class"].value_counts().describe())

print("\nClasses missing from new dataset:",
      sorted(source_classes - new_classes))

print("Unexpected class labels:",
      sorted(new_classes - source_classes))

print("\nOverlap with original training data:", len(train_overlap))
print("Overlap with original testing data:", len(test_overlap))

print("\nMissing values:")
display(independent_test_df[required_columns].isna().sum())

print("\nPhrases matching the original training data:")
display(train_overlap[["phrase", "expected_class"]])

print("\nPhrases matching the original testing data:")
display(test_overlap[["phrase", "expected_class"]])

Saving WasteLoop_Independent_Test_250.csv to WasteLoop_Independent_Test_250.csv
INDEPENDENT TEST DATA AUDIT
---------------------------------------------
Total examples: 250
Unique classes: 50
Duplicate phrases within new dataset: 0

Class count distribution:


,count
count,50.0
mean,5.0
std,0.0
min,5.0
25%,5.0
50%,5.0
75%,5.0
max,5.0



Classes missing from new dataset: []
Unexpected class labels: []

Overlap with original training data: 0
Overlap with original testing data: 0

Missing values:


,0
test_id,0
phrase,0
expected_class,0
case_type,0
expected_action,0



Phrases matching the original training data:


,phrase,expected_class



Phrases matching the original testing data:


,phrase,expected_class


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# I am evaluating my trained baseline on independently prepared examples.
X_independent = independent_test_df["phrase"]
y_independent = independent_test_df["expected_class"]

independent_predictions = tfidf_model.predict(X_independent)

# I am calculating the evaluation metrics.
ind_accuracy = accuracy_score(
    y_independent, independent_predictions
)

ind_precision = precision_score(
    y_independent, independent_predictions,
    average="macro", zero_division=0
)

ind_recall = recall_score(
    y_independent, independent_predictions,
    average="macro", zero_division=0
)

ind_f1 = f1_score(
    y_independent, independent_predictions,
    average="macro", zero_division=0
)

ind_weighted_f1 = f1_score(
    y_independent, independent_predictions,
    average="weighted", zero_division=0
)

print("INDEPENDENT TEST RESULTS")
print("-" * 45)
print(f"Accuracy:        {ind_accuracy:.4f}")
print(f"Macro Precision: {ind_precision:.4f}")
print(f"Macro Recall:    {ind_recall:.4f}")
print(f"Macro F1:        {ind_f1:.4f}")
print(f"Weighted F1:     {ind_weighted_f1:.4f}")

print("\nDetailed classification report:")
print(classification_report(
    y_independent,
    independent_predictions,
    zero_division=0
))

INDEPENDENT TEST RESULTS
---------------------------------------------
Accuracy:        0.5160
Macro Precision: 0.5846
Macro Recall:    0.5160
Macro F1:        0.5010
Weighted F1:     0.5010

Detailed classification report:
                               precision    recall  f1-score   support

                ALUMINIUM CAN       0.33      0.60      0.43         5
              ALUMINIUM SHEET       0.67      0.40      0.50         5
               ALUMINIUM TRAY       1.00      0.20      0.33         5
               CIGARETTE BUTT       0.67      0.80      0.73         5
               CIGARETTE PACK       0.00      0.00      0.00         5
        COMPOSTABLE PACKAGING       0.00      0.00      0.00         5
            CONDIMENT PACKETS       0.67      0.80      0.73         5
                   COVID TEST       1.00      1.00      1.00         5
              CRUMBLED TISSUE       0.56      1.00      0.71         5
          CYLINDRICAL BATTERY       0.50      0.20      0.29     

In [ ]:
!pip -q install sentence-transformers

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# I am loading a pretrained sentence embedding model.
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# I am converting my training, validation and independent test text into embeddings.
X_train_embeddings = embedding_model.encode(
    X_train.astype(str).tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

X_val_embeddings = embedding_model.encode(
    X_val.astype(str).tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

X_independent_embeddings = embedding_model.encode(
    independent_test_df["phrase"].astype(str).tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

# I am training a classifier on the sentence embeddings.
embedding_classifier = LogisticRegression(
    max_iter=2000,
    random_state=42
)

embedding_classifier.fit(X_train_embeddings, y_train)

# I am evaluating the model on both datasets.
val_predictions = embedding_classifier.predict(X_val_embeddings)

independent_predictions_embedding = embedding_classifier.predict(
    X_independent_embeddings
)

# I am comparing the results with my TF-IDF baseline.
comparison_df = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "Sentence Embeddings + Logistic Regression"
    ],
    "Validation Accuracy": [
        accuracy,
        accuracy_score(y_val, val_predictions)
    ],
    "Independent Test Accuracy": [
        ind_accuracy,
        accuracy_score(
            independent_test_df["expected_class"],
            independent_predictions_embedding
        )
    ],
    "Validation Macro F1": [
        f1_macro,
        f1_score(y_val, val_predictions, average="macro")
    ],
    "Independent Test Macro F1": [
        ind_f1,
        f1_score(
            independent_test_df["expected_class"],
            independent_predictions_embedding,
            average="macro"
        )
    ]
})

display(comparison_df.round(4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (7 outputs left out)

Batches:   0%|          | 0/251 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

,Model,Validation Accuracy,Independent Test Accuracy,Validation Macro F1,Independent Test Macro F1
0,TF-IDF + Logistic Regression,0.9878,0.516,0.9844,0.501
1,Sentence Embeddings + Logistic Regression,0.9855,0.724,0.9819,0.720


In [ ]:
import os
import json
import joblib
import zipfile
from sklearn.metrics import accuracy_score, f1_score
from google.colab import files

# I am saving my selected sentence embedding model.
output_dir = "/content/WasteLoop_Sentence_Embedding_Model"
os.makedirs(output_dir, exist_ok=True)

# I am saving the pretrained embedding model.
embedding_model.save(
    f"{output_dir}/sentence_transformer"
)

# I am saving the trained classifier.
joblib.dump(
    embedding_classifier,
    f"{output_dir}/logistic_regression.joblib"
)

# I am saving the model evaluation results.
results = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "classifier": "Logistic Regression",
    "validation_accuracy": 0.9855,
    "independent_test_accuracy": 0.7240,
    "independent_test_macro_f1": 0.7200,
    "training_samples": len(X_train),
    "validation_samples": len(X_val),
    "independent_test_samples": len(independent_test_df),
    "number_of_classes": 50
}

with open(f"{output_dir}/evaluation_results.json", "w") as f:
    json.dump(results, f, indent=4)

# I am saving the comparison with my TF-IDF baseline.
comparison_df.to_csv(
    f"{output_dir}/model_comparison.csv",
    index=False
)

# I am packaging all saved files into one ZIP.
zip_path = "/content/WasteLoop_Sentence_Embedding_Model.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, filenames in os.walk(output_dir):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            z.write(
                filepath,
                os.path.relpath(filepath, output_dir)
            )

print("Sentence embedding model saved successfully.")
print("Independent accuracy:", 72.40, "%")

files.download(zip_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sentence embedding model saved successfully.
Independent accuracy: 72.4 %


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import joblib
import pickle
from google.colab import drive

# I am connecting my Google Drive.
drive.mount('/content/drive')

# I am creating a folder for my WasteWise models.
model_dir = "/content/drive/MyDrive/WasteWise/Models"
os.makedirs(model_dir, exist_ok=True)

# I am saving my Sentence Transformer model.
embedding_model.save(
    os.path.join(model_dir, "sentence_transformer")
)

# I am saving my Logistic Regression classifier.
joblib.dump(
    embedding_classifier,
    os.path.join(model_dir, "sentence_classifier.pkl")
)

# I am saving my TF-IDF baseline as well.
joblib.dump(
    tfidf_model,
    os.path.join(model_dir, "tfidf_baseline.pkl")
)

# I am saving the model comparison results.
with open(
    os.path.join(model_dir, "model_comparison.pkl"), "wb"
) as f:
    pickle.dump(comparison_df, f)

print("All models and evaluation results saved to Google Drive.")
print("Location:", model_dir)

ValueError: mount failed

In [ ]:
import re
import pandas as pd

# I am extracting basic information from the user's description.

def extract_waste_information(text):

    text_lower = text.lower()

    # I am identifying the quantity mentioned by the user.
    quantity_match = re.search(
        r'\b(\d+(?:\.\d+)?)\s*(kg|kgs|kilograms?|grams?|g|'
        r'litres?|liters?|ml|pieces?|pcs|items?|bottles?|'
        r'packets?|bags?|cans?)\b',
        text_lower
    )

    if quantity_match:
        quantity = quantity_match.group(1)
        quantity_unit = quantity_match.group(2)
    else:
        count_match = re.search(
            r'\b(\d+(?:\.\d+)?)\s+(?=(?:empty|used|clean|'
            r'dirty|broken|old|plastic|paper|glass|metal)\b)',
            text_lower
        )
        quantity = count_match.group(1) if count_match else None
        quantity_unit = "items (inferred)" if count_match else None

    # I am identifying the item's condition.
    condition_keywords = [
        "empty", "clean", "dirty", "washed", "rinsed",
        "dry", "wet", "broken", "damaged", "used", "intact"
    ]

    condition = [
        word for word in condition_keywords
        if re.search(r'\b' + word + r'\b', text_lower)
    ]

    # I am identifying what the user wants to do with the waste.
    intent_keywords = {
        "sell": ["sell", "selling", "scrap", "earn money"],
        "donate": ["donate", "donation", "give away"],
        "recycle": ["recycle", "recycling"],
        "reuse": ["reuse", "upcycle", "repurpose"],
        "dispose": ["dispose", "throw away", "discard"]
    }

    intents = []

    for intent, keywords in intent_keywords.items():
        if any(keyword in text_lower for keyword in keywords):
            intents.append(intent)

    # I am identifying the waste category using my trained model.
    embedding = embedding_model.encode(
        [text],
        normalize_embeddings=True
    )

    predicted_class = embedding_classifier.predict(embedding)[0]

    model_score = embedding_classifier.predict_proba(
        embedding
    ).max()

    return {
        "description": text,
        "predicted_class": predicted_class,
        "quantity": quantity,
        "quantity_unit": quantity_unit,
        "condition": condition,
        "user_intent": intents,
        "model_score": round(float(model_score), 4)
    }


# I am testing my information extraction function.
sample_input = (
    "I have 3 empty plastic bottles. "
    "They are clean and I want to recycle them."
)

result = extract_waste_information(sample_input)

display(pd.DataFrame([result]))

In [ ]:
# I am listing the DataFrames available in this notebook.

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(name, obj.shape, list(obj.columns))

train_df (21528, 8) ['Unnamed: 0', 'Phrase', 'Class', 'Class_index', 'normalized_phrase', 'word_count', 'opening_pattern', 'Phrase_norm']
test_df (5382, 7) ['Unnamed: 0', 'Phrase', 'Class', 'Class_index', 'normalized_phrase', 'opening_pattern', 'opening_seen_in_train']
working_df (21528, 9) ['Unnamed: 0', 'Phrase', 'Class', 'Class_index', 'normalized_phrase', 'word_count', 'opening_pattern', 'Phrase_norm', 'phrase_key']
conflict_quarantine_df (821, 8) ['Unnamed: 0', 'Phrase', 'Class', 'Class_index', 'normalized_phrase', 'word_count', 'opening_pattern', 'Phrase_norm']
consistent_df (20707, 9) ['Unnamed: 0', 'Phrase', 'Class', 'Class_index', 'normalized_phrase', 'word_count', 'opening_pattern', 'Phrase_norm', 'phrase_key']
train_clean (20002, 8) ['Unnamed: 0', 'Phrase', 'Class', 'Class_index', 'normalized_phrase', 'word_count', 'opening_pattern', 'Phrase_norm']
error_df (49, 5) ['Phrase', 'Actual', 'Predicted', 'Predicted_confidence', 'Actual_class_probability']
confusion_pairs (6, 3) ['